# Burgers: example predictions and the test error distribution

We learn the solution operator $K: u_0 \mapsto u(\cdot, T)$ of the viscous Burgers' equation
$u_t + u u_x = \nu u_{xx}$ on $(0,1)$, $u(0,t) = u(1,t) = 0$, with $T = 0.2$.

* Inputs and outputs are sine coefficients in the basis $\sqrt 2 \sin(n\pi x)$; data come from the spectral Galerkin solver in `burgers_solver.py`.
* Input measure: $d_{\mathrm{in}} = 20$ independent coefficients, the $n$-th distributed as $\mathrm{Jacobi}(\alpha_n, \alpha_n)$. **As run:** $\alpha_n = (n-1)^2$, $n = 1, \dots, 20$ (so the first mode is uniform on $[-1,1]$).
* Approximation space: polynomial operators $\psi_m\, p_\lambda(u_0)$ with $\lambda$ in an anisotropic index set with weights $w_j$ = `linspace(1, 0.01, 20)` (the weights enter as an exponent $1/w_j$, see `owls.index_set`) and univariate degree at most 10.

Here $\nu = 0.1$ and the index set is the weighted hyperbolic cross with $k = 60$ ($N_{\mathrm{eff}} = 5046$), fit from $M = \lfloor N \log N\rfloor$ optimally sampled inputs. We show the test inputs with the largest, median and smallest error out of 1000, and the distribution of the test errors.

**Status:** the figure in `figures/` is from the arXiv v1 run. That run did not save its data, so this notebook can only redraw the figure after `RECOMPUTE = True` (a new random draw; takes a few minutes).

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
import seaborn as sns
from burgers_solver import burgers_map, sine_basis_vandermonde

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
NU, T, DT = 0.1, 0.2, 1e-3
D_OUT = 150            # output modes
D_SOLVE = 150          # as run: the solver uses max(5 * D_IN, D_OUT) = 150 modes
HC_K = 60
MAX_SIZE = 10000
M_TEST = 1000
SEED = 0
if QUICK:
    HC_K, M_TEST, T = 5, 50, 0.01

D_IN = 20                                             # input sine modes
MAX_DEGREE = 10                                       # max univariate polynomial degree
measures = [owls.Jacobi(a**2) for a in range(D_IN)]  # as run: alpha_n = (n - 1)^2 for n = 1..20
weights = np.linspace(1, 0.01, D_IN)                 # anisotropy weights

In [ ]:
DATA_FILE = os.path.join(DATA, "example_solutions.npz")
if RECOMPUTE or QUICK:
    rng = np.random.default_rng(SEED)
    L = owls.index_set(D_IN, HC_K, "hc", weights, MAX_DEGREE, MAX_SIZE)
    N = len(L)
    solver = lambda u0: burgers_map(u0, NU, T, DT, D_SOLVE, D_OUT)
    model = owls.learn_operator(solver, measures, L, M=int(N * np.log(N)), optimal=True, rng=rng)
    x_test = owls.sample_rho(M_TEST, measures, rng)
    y_test = solver(x_test)
    y_pred = model.predict(x_test)
    test_errors = np.linalg.norm(y_pred - y_test, axis=1)
    idx = [np.argmax(test_errors), np.argmin(np.abs(np.median(test_errors) - test_errors)), np.argmin(test_errors)]
    np.savez(DATA_FILE, test_errors=test_errors, u0=x_test[idx], true=y_test[idx], pred=y_pred[idx])
    print(f"N_eff = {N}, M = {model.M}, cond(G) = {model.cond():.2f}")

In [ ]:
if not os.path.exists(DATA_FILE):
    print("No saved data: see figures/vB_err_dist.pdf for the arXiv v1 figure, or set RECOMPUTE = True.")
else:
    dat = np.load(DATA_FILE)
    test_errors = dat["test_errors"]
    boch_test_err = np.sqrt(np.mean(test_errors**2))
    x = np.linspace(0, 1, 1000)
    Vi = np.sqrt(2) * sine_basis_vandermonde(x, dat["u0"].shape[1])
    Vo = np.sqrt(2) * sine_basis_vandermonde(x, dat["true"].shape[1])

    fig_width, fig_height = paper_style(width=1.0, aspect=0.4, fontsize=7, legend_fontsize=5)
    fig, ax = plt.subplots(ncols=4, figsize=(fig_width, fig_height), layout="constrained")
    titles = [r"Max $L^2_{\mathcal{Y}}$ Error: ", r"Median $L^2_{\mathcal{Y}}$ Error: ", r"Min $L^2_{\mathcal{Y}}$ Error: "]
    for a in range(3):
        ax[a].plot(x, Vi @ dat["u0"][a], "k:", label=r"$u_0$", linewidth=0.4)
        ax[a].plot(x, Vo @ dat["true"][a], "k-", label=r"$K(u_0)$", linewidth=0.9)
        ax[a].plot(x, Vo @ dat["pred"][a], "r--", label=r"$\tilde{K}_V(u_0)$", linewidth=0.9)
        err = np.linalg.norm(dat["pred"][a] - dat["true"][a])
        ax[a].set_title(titles[a] + f"{err:.2e}")
        ax[a].set_xlabel(r"$x$")
        ax[a].legend(loc="lower left" if a != 1 else "best")
    ax[3].hist(test_errors, bins=max(10, len(test_errors) // 7), density=True, color="black", alpha=0.1)
    sns.kdeplot(test_errors, fill=True, color="black", cut=0, ax=ax[3], linewidth=0.9)
    ax[3].set_xlabel(r"$L^2_{\mathcal{Y}}$ Error")
    ax[3].set_title(r"$L^2_\rho(\mathcal{X}; \mathcal{Y})$ Error: " + f"{boch_test_err:.2e}")
    sns.despine(trim=True)
    savefig(os.path.join(FIGS, "vB_err_dist.pdf"))
    plt.show()